# Texto → SQL com LLM local (QLoRA em GPU)

Notebook de treino e avaliação do trabalho. Fluxo:

1. Verificar a GPU (print exigido no trabalho)
2. Carregar e entender os dados
3. Avaliar o **modelo base** (zero-shot) — ponto de comparação
4. **Treinar** com QLoRA
5. **Avaliar** o modelo fine-tuned (exact match, SQL válido, acurácia de execução)
6. **Testar parâmetros** (rank do LoRA, learning rate, épocas, temperature/top_p)
7. Subir o **frontend** (Gradio)

> Execute as células em ordem, de dentro do VSCode, com o painel de monitoramento da GPU aberto.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
# Se ainda não instalou as dependências (instale o PyTorch com CUDA ANTES: veja docs/GPU_CUDA.md)
# !pip install -r requirements.txt

## 1. GPU e CUDA

Tire o print **desta célula** com o painel do GPU Monitor / GPU Environment visível.

In [ ]:
from src.gpu import print_gpu_report
print_gpu_report()

In [ ]:
!nvidia-smi

## 2. Dados

Dataset: [`b-mc2/sql-create-context`](https://huggingface.co/datasets/b-mc2/sql-create-context) — triplas
*(pergunta, CREATE TABLE, SQL)*. O esquema vai junto no prompt para o modelo não inventar tabelas/colunas.
Misturamos também exemplos em **português** escritos pelo grupo (`data/pt_br_train.jsonl`).

In [ ]:
from src.data import load_sql_dataset, load_pt_examples, load_eval_examples

data = load_sql_dataset(train_samples=3000, eval_samples=200, seed=42, pt_repeat=3)
print("treino:", len(data["train"]), "| avaliação (EN):", len(data["eval"]),
      "| avaliação (PT):", len(load_pt_examples("eval")))
for ex in data["train"][:3]:
    print(ex, "\n")

### Como o prompt é montado

In [ ]:
from transformers import AutoTokenizer
from src.config import DEFAULT_MODEL
from src.prompts import render_prompt

tok = AutoTokenizer.from_pretrained(DEFAULT_MODEL)
ex = data["eval"][0]
print(render_prompt(tok, ex["context"], ex["question"]) + ex["answer"])

## 3. Baseline: modelo base sem fine-tuning

Métricas: **exact match** (texto igual após normalização), **SQL válido** (executa sem erro) e
**acurácia de execução** (o resultado é igual ao da consulta de referência em bancos SQLite com dados fictícios).

In [ ]:
from src.config import GenConfig
from src.inference import SQLGenerator
from src.evaluate import evaluate_generator

gen = SQLGenerator.load(DEFAULT_MODEL, adapter_path=None, use_4bit=True)
m_base, rows_base = evaluate_generator(gen, data["eval"], GenConfig(), desc="modelo base")
m_base_pt, _ = evaluate_generator(gen, load_pt_examples("eval"), GenConfig(), desc="modelo base (PT)")
print(m_base)
print(m_base_pt)
gen.free()   # libera a VRAM antes do treino

## 4. Treino (QLoRA)

Hiperparâmetros principais (todos ajustáveis em `TrainConfig`): `lora_r`, `lora_alpha`, `learning_rate`,
`epochs`, `batch_size × grad_accum`, `train_samples`. Se faltar VRAM: reduza `batch_size` (e aumente `grad_accum`).

In [ ]:
from src.config import TrainConfig
from src.train import run_training

cfg = TrainConfig(run_name="baseline", epochs=1, learning_rate=2e-4, lora_r=16, lora_alpha=32,
                  batch_size=4, grad_accum=4, train_samples=3000)
info = run_training(cfg, data=data)
{k: v for k, v in info.items() if k != "log_history"}

In [ ]:
import matplotlib.pyplot as plt

logs = [e for e in info["log_history"] if "loss" in e]
plt.plot([e["step"] for e in logs], [e["loss"] for e in logs])
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Curva de loss (treino)"); plt.grid(alpha=.3)
plt.show()

## 5. Avaliação do modelo fine-tuned

In [ ]:
import pandas as pd

gen = SQLGenerator.load(DEFAULT_MODEL, adapter_path=str(cfg.adapter_dir), use_4bit=True)
m_ft, rows_ft = evaluate_generator(gen, data["eval"], GenConfig(), desc="fine-tuned")
m_ft_pt, rows_ft_pt = evaluate_generator(gen, load_pt_examples("eval"), GenConfig(), desc="fine-tuned (PT)")

pd.DataFrame({"base (EN)": m_base, "fine-tuned (EN)": m_ft,
              "base (PT)": m_base_pt, "fine-tuned (PT)": m_ft_pt}).T

### Olhando as respostas (qualitativo)

In [ ]:
df = pd.DataFrame(rows_ft)
df[["question", "gold", "pred", "exact_match", "exec_match"]].head(10)

In [ ]:
# Erros: onde o modelo não acertou a execução
df[df["exec_match"] == False][["question", "gold", "pred"]].head(10)

In [ ]:
# Teste livre: base x fine-tuned, em português
schema = "CREATE TABLE alunos (id INTEGER, nome VARCHAR, turma VARCHAR, nota FLOAT)"
pergunta = "Quais alunos da turma 4ESPG tiveram nota acima de 8?"
print("base      :", gen.generate_one(schema, pergunta, GenConfig(), use_adapter=False))
print("fine-tuned:", gen.generate_one(schema, pergunta, GenConfig(), use_adapter=True))

In [ ]:
gen.free()

## 6. Testando diferentes parâmetros

A varredura roda **um parâmetro por vez** em relação ao baseline:

| Parâmetro | Valores |
|---|---|
| Rank do LoRA (`lora_r`, com `alpha = 2r`) | 4, **16**, 64 |
| Learning rate | 5e-5, **2e-4**, 5e-4 |
| Épocas | **1**, 2, 3 |
| `temperature` / `top_p` (inferência) | greedy, 0.3, 0.7, 1.0 × 0.9, 1.0 |

Leva de ~30 a ~90 min dependendo da GPU. Faça antes um teste rápido com `quick=True`.

In [ ]:
RUN_SWEEP = False   # troque para True para rodar a varredura completa
QUICK = False       # True = versão reduzida só para testar o pipeline

if RUN_SWEEP:
    from src.experiments import run_train_sweep, run_inference_sweep
    run_train_sweep(train_samples=2000, eval_samples=200, quick=QUICK)
    adapter = ROOT / "outputs" / ("quick_baseline" if QUICK else "baseline") / "adapter"
    run_inference_sweep(str(adapter), eval_samples=60 if QUICK else 200)

In [ ]:
import pandas as pd
pd.read_csv(ROOT / "results" / "experiments.csv")

In [ ]:
!python scripts/plot_results.py

In [ ]:
from IPython.display import Image, display
for name in ["loss_curves", "experiments", "inference_sweep"]:
    p = ROOT / "docs" / "img" / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 7. Frontend

No terminal do VSCode (na raiz do projeto):

```bash
python app/app.py                                  # usa outputs/baseline/adapter
python app/app.py --adapter outputs/r64/adapter    # ou outro adaptador
```

Abra http://127.0.0.1:7860 — escolha um banco, faça uma pergunta e veja o SQL e o resultado.
Para uma **demonstração com o modelo desta sessão**, feche-o antes (`gen.free()`) para liberar a VRAM.

## 8. Conclusões (preencher com os resultados do grupo)

- **Melhoria do fine-tuning:** base → fine-tuned (exec. accuracy): `__` → `__`
- **Melhor configuração:** `__` (r, lr, épocas) e por quê
- **Efeito de temperature/top_p:** `__`
- **Principais erros encontrados:** `__` (ex.: JOINs, agregações, literais em português)
- **Limitações e próximos passos:** `__`